[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giswqs/GEE-Courses/blob/master/docs/gee_intro/AssetManagement/export_data.ipynb)

In [ ]:
#AKB NOTES: extensive modification from original file. Tests of tif and png exports. See "CONCLUSION" notes below. 
#   Upshot is use geemap_AKB... everything useful from here is now there too.

#Test outputs are here: C:\Users\andyb\Documents\U\GEE-Courses\data and shown in test_geemap.qgz
#See Grok "python geemap export jpg" for additional ideas.
#See also: geemap_AKB.ipynb and test_proj.ipynb

# !pip install geemap

In [ ]:
import os
import ee
import geemap
from geemap.datasets import DATA
Map = geemap.Map()

## Set an output directory

In [ ]:
out_dir = os.path.expanduser("~/Downloads")

if not os.path.exists(out_dir):
    os.makedirs(out_dir)

## Export an ee.FeatureCollection

In [ ]:
fc = ee.FeatureCollection('users/giswqs/public/countries')

In [ ]:
Map.addLayer(fc, {}, "Countries")
Map.centerObject(fc)
Map

Export data to a local computer

In [ ]:
out_shp = os.path.join(out_dir, "countries.shp")
#geemap.ee_export_vector(fc, out_shp, verbose=True)

Export data to Google Drive

In [ ]:
#geemap.ee_export_vector_to_drive(fc, description="countries", folder="export")

## Export an ee.Image

In [ ]:
image = ee.Image('LE7_TOA_5YEAR/1999_2003') #uses 3,2,1 as true color

landsat_vis = {'bands': ['B4', 'B3', 'B2'], 'gamma': 1.7}
Map.addLayer(image, landsat_vis, "LE7_TOA_5YEAR/1999_2003", True, 1)
Map

In [ ]:
#test with single image too, not just 5yr composite
imageSF = ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_044034_20140318')
Map.addLayer(imageSF, landsat_vis, "SF", True, 1)
Map

In [ ]:
out_dir = os.path.join(os.path.expanduser('~'), 'Downloads')
filename = os.path.join(out_dir, 'landsat2.tif')

### Exporting all bands as one single image

In [ ]:
# set a default ROI
roi = ee.Geometry.Polygon(
    [
        [
#            [-482.643127, 37.547027], #example
#            [-482.643127, 37.996433],
#            [-481.908417, 37.996433],
#            [-481.908417, 37.547027],
#            [-482.643127, 37.547027],
            [-138.017076, 58.59777], #Lituya
            [-138.017076, 58.87785],
            [-137.2163, 58.87785],
            [-137.2163, 58.59777],
            [-138.017076, 58.59777]
        ]
    ]
)

roiSF = ee.Geometry.Polygon(
    [
        [
            [-122.553199, 37.746027],
            [-122.553199, 37.841254],
            [-122.344763, 37.841254],
            [-122.344763, 37.746027],
            [-122.553199, 37.746027]
        ]
    ]
)

In [ ]:
# Draw any shapes on the map using the Drawing tools before executing this code block
if Map.user_roi is not None:
    roi = Map.user_roi

In [ ]:
roi.getInfo()

In [ ]:
image_clip = image.clip(roi)
image_clipSF = imageSF.clip(roiSF)

In [ ]:
Map.addLayer(image_clip, landsat_vis, "image")
Map.addLayer(image_clipSF, landsat_vis, "imageSF")

In [ ]:
#export .tif
geemap.ee_export_image( #only .tif
    image_clip, filename=filename, region=roi, file_per_band=False # scale=30, 
)
#CONCLUSION: scale in m, 30 or 90. What happens without it? Keeps best resolution.
#CONCLUSION: actual pixels are 15x30 or 45x90
#CONCLUSION: CRS of these tifs is EPSG:4326 WGS84

In [ ]:
#geemap.ee_export_image(
#    image_clip, filename=os.path.join(out_dir, 'landsatFAIL.png'), scale=30, region=roi, file_per_band=False #scale in m, 30 or 90
#)
#ERROR: The filename must end with .tif

In [ ]:
# Export the layer to JPG (from Grok)
Map.layer_to_image(layer_name='image', output= os.path.join(out_dir, 'landsat2_scale30.png'), scale=30, region=roi) #
Map.layer_to_image(layer_name='image', output= os.path.join(out_dir, 'landsat2_scale90.png'), scale=90, region=roi) #
Map.layer_to_image(layer_name='image', output= os.path.join(out_dir, 'landsat2_scaleNone.png'), region=roi) #
#CONCLUSION: at scale 90: this is in WGS84 projection and doesn't look good. Shearing along pixel boundaries (tif also in WGS but not skewed).
#CONCLUSION: MIGHT be good enough at 30, depending on use case or specific glacier
#CONCLUSION: without scale, it's a small thumbnail >90 m pixels or super small with next run through


In [ ]:
#SF
geemap.ee_export_image( #only .tif
    image_clipSF, filename=os.path.join(out_dir, 'landsatSF_scale30.tif'), region=roiSF, file_per_band=False, scale=30
)
#CONCLUSION: export is in UTM10N. 

In [ ]:
Map.layer_to_image(layer_name='imageSF', output= os.path.join(out_dir, 'landsatSF_scale30.png'), scale=30, region=roiSF) #
#CONCLUSION: MIGHT be good enough at 30, depending on use case - see distortions in Bay Bridge, but city not bad

In [ ]:
#another way??
geemap.get_image_thumbnail(image_clip, os.path.join(out_dir, 'landsat3x1000.png'), landsat_vis, dimensions=1000)
geemap.get_image_thumbnail(image_clip, os.path.join(out_dir, 'landsat3.png'), landsat_vis, dimensions=2000)
#CONCLUSION: seems reasonable - same problem with skew as pngs above. Nice/not nice that dimensions are specified instead of resolution

In [ ]:
geemap.get_image_thumbnail(image_clip, os.path.join(out_dir, 'landsat3albers.png'), landsat_vis, dimensions=2000, crs='EPSG:3338') #Alaska Albers as a test
geemap.get_image_thumbnail(image_clip, os.path.join(out_dir, 'landsat3utm8N.png'), landsat_vis, dimensions=2000, crs='EPSG:32608') #UTM 8N
#CONCLUSION: this solves skew at the expense of rotation/black corners of png - is it possible to apply clip with UTM roi?

In [ ]:
geemap.show_image(os.path.join(out_dir, 'landsat3.png'))

In [ ]:
#Map.to_image
#Make sure you click the fullscreen button on the map to maximum the map.
#https://geemap.org/notebooks/21_export_map_to_html_png/#exporting-maps-as-html
png_file = os.path.join(out_dir, "my_map.png")
Map.to_image(filename=png_file, monitor=1)
#jpg_file = os.path.join(out_dir, "my_map.jpg")
#Map.to_image(filename=jpg_file, monitor=1)
#CONCLUSION: this essentially takes a screenshot, not helpful because it shows code rather than map.

### Exporting each band as one image

In [ ]:
geemap.ee_export_image(
    image, filename=filename, scale=90, region=roi, file_per_band=True
)

### Export an image to Google Drive

In [ ]:
#geemap.ee_export_image_to_drive(
#    image, description='landsat', folder='export', region=roi, scale=30
#)

## Download an ee.ImageCollection (NAIP)

In [ ]:
loc = ee.Geometry.Point(-99.2222, 46.7816) #near North Dakota/South Dakota border
collection = (
    ee.ImageCollection('USDA/NAIP/DOQQ')
    .filterBounds(loc)
    .filterDate('2008-01-01', '2020-01-01')
    .filter(ee.Filter.listContains("system:band_names", "N"))
)

In [ ]:
out_dir = os.path.expanduser('~/Downloads')

In [ ]:
labelsNAIP = collection.aggregate_array('system:index').getInfo()
labelsNAIP

In [ ]:
Map.add_time_slider(collection, {}, labels=labelsNAIP, time_interval=1) #AKB added to visualize


In [ ]:
#geemap.ee_export_image_collection(collection, out_dir=out_dir)
#An error occurred while downloading.
#Total request size (822703200 bytes) must be less than or equal to 50331648 bytes.

In [ ]:
#therefor, AKB added code to select area:
# Draw a small rectangle on the map using the Drawing tools before executing this code block
if Map.user_roi is not None:
    roiND = Map.user_roi
roiND.getInfo()

In [ ]:
roiND = ee.Geometry.Polygon( #slightly different than original ROI
    [
        [
            [-99.224132, 46.778977],
            [-99.224132, 46.786088],
            [-99.215977, 46.786088],
            [-99.215977, 46.778977],
            [-99.224132, 46.778977]
        ]
    ]
)

In [ ]:
#collection_clip = collection.clip(roi)
#FAILED since clip only works on images

# Function to clip each image in the collection to the ROI
def clip_image(image):
    return image.clip(roiND)

# Apply clipping to the entire collection
collection_clip = collection.map(clip_image)

In [ ]:
geemap.ee_export_image_collection(collection_clip, out_dir=out_dir)

In [ ]:
#geemap.ee_export_image_collection_to_drive(collection, folder='export', scale=10)

In [ ]:
vis_params = {
    "bands": ["R", "G", "B"],
#    "min": 0,
#    "max": 6000,
#    "gamma": 1.4,
}
geemap.get_image_collection_thumbnails(collection_clip, out_dir, vis_params, dimensions=500, format="png")
#CONCLUSION: works - same projection as display (~WGS84)

In [ ]:
#FAILS: geemap.get_image_collection_thumbnails(collection_clip, out_dir, vis_params, dimensions=500, format="png", crs='EPSG:3338') #Alaska Albers as a test
#BECAUSE collection_thumbs doesn't take crs, so define function that does it one by one...

# Get list of image IDs in the collection
#image_list = collection_clip.aggregate_array('system:id').getInfo()

# Loop through each image and generate thumbnail
#for image_id in image_list:
#    try:
#        # Load the image - AKB note: shouldn't need to reload - this gets full image instead of clipped one
#        image = ee.Image(image_id)        
#        # Define output filename (use image ID or custom name)
#        filename = os.path.join(out_dir, f'{image_id.replace("/", "_")}.png')
#        # Generate thumbnail
#        geemap.get_image_thumbnail(image,filename,vis_params=vis_params,dimensions=2000,crs='EPSG:3338') #Alaska Albers as a test
#        print(f'Thumbnail generated for {image_id}: {filename}')
#    except Exception as e:
#        print(f'Error processing {image_id}: {e}')
#
#print('Thumbnail generation complete!')

In [ ]:
#Convert collection to a list to iterate over images
image_list = collection_clip.toList(collection_clip.size())

# Get collection size
size = collection_clip.size().getInfo()
size

In [ ]:
#Export png
#Loop through each image in the collection
for i in range(size):
    try:
        # Get the image from the list
        image = ee.Image(image_list.get(i))
        
        # Get image ID for filename
        image_id = image.get('system:id').getInfo()
        
        # Define output filename
        filename = os.path.join(out_dir, f'{image_id.replace("/", "_")}.png') #e.g. USDA/NAIP/DOQQ/m_4609915_sw_14_h_20170703
        
        # Generate thumbnail
        geemap.get_image_thumbnail(image,filename,vis_params=vis_params,dimensions=2000,crs='EPSG:3338') #Alaska Albers as a test
        print(f'Thumbnail generated for {image_id}: {filename}')
    except Exception as e:
        print(f'Error processing image {i}: {e}')

print('Thumbnail generation complete!')

## Extract pixels as a Numpy array

In [ ]:
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt

img = ee.Image('LANDSAT/LC08/C02/T1_L2/LC08_038029_20180810').select(['SR_B4', 'SR_B5', 'SR_B6'])

aoi = ee.Geometry.Polygon(
    [[[-110.8, 44.7], [-110.8, 44.6], [-110.6, 44.6], [-110.6, 44.7]]], None, False
)

rgb_img = geemap.ee_to_numpy(img, region=aoi)
print(rgb_img.shape)

In [ ]:
# Scale the data to [0, 255] to show as an RGB image.
# Adapted from https://bit.ly/2XlmQY8. Credits to Justin Braaten
rgb_img_test = (255 * ((rgb_img[:, :, 0:3] - 100) / 3500)).astype('uint8')
plt.imshow(rgb_img_test)
plt.show()